In [1]:
from sentence_transformers import SentenceTransformer

# 这个是embedding模型
model = SentenceTransformer('DMetaSoul/Dmeta-embedding')


In [ ]:
from sentence_transformers.util import semantic_search

hits = semantic_search(query_embeddings, dataset_embeddings, top_k=5)

In [4]:

texts1 = ["去打开空空"]
texts2 = ["开启空调", "帮我打开空调", "打开空调", "空调打开", "不打开空调", "关闭空调", "空调关上", "启动一下空调", "把空调打开"]

embs1 = model.encode(texts1, normalize_embeddings=True)
embs2 = model.encode(texts2, normalize_embeddings=True)

similarity = embs1 @ embs2.T
print(similarity)

for i in range(len(texts1)):
    scores = []
    for j in range(len(texts2)):
        scores.append([texts2[j], similarity[i][j]])
    scores = sorted(scores, key=lambda x:x[1], reverse=True)

    print(f"查询文本：{texts1[i]}")
    for text2, score in scores:
        print(f"相似文本：{text2}，打分：{score}")
    print()

[[0.5084244  0.5197594  0.55420804 0.54226935 0.41972697 0.38920665
  0.42130357 0.49159047 0.57482815]]
查询文本：去打开空空
相似文本：把空调打开，打分：0.5748281478881836
相似文本：打开空调，打分：0.5542080402374268
相似文本：空调打开，打分：0.5422693490982056
相似文本：帮我打开空调，打分：0.5197594165802002
相似文本：开启空调，打分：0.5084244012832642
相似文本：启动一下空调，打分：0.4915904700756073
相似文本：空调关上，打分：0.42130357027053833
相似文本：不打开空调，打分：0.4197269678115845
相似文本：关闭空调，打分：0.3892066478729248



In [7]:
type(embs1)
import numpy as np
row_norms = np.linalg.norm(embs1, axis=1)  # 按行计算范数
print(row_norms)  # 输出: [5. 2.23606798]

[1. 1.]


In [ ]:
# Requires transformers>=4.51.0
# Requires sentence-transformers>=2.7.0

from sentence_transformers import SentenceTransformer

# Load the model
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")

# We recommend enabling flash_attention_2 for better acceleration and memory saving,
# together with setting `padding_side` to "left":
# model = SentenceTransformer(
#     "Qwen/Qwen3-Embedding-0.6B",
#     model_kwargs={"attn_implementation": "flash_attention_2", "device_map": "auto"},
#     tokenizer_kwargs={"padding_side": "left"},
# )


In [6]:

# The queries and documents to embed
queries = [
    "What is the capital of China?",
    "Explain gravity",
]
documents = [
    "The capital of China is Beijing.",
    "Gravity is a force that attracts two bodies towards each other. It gives weight to physical objects and is responsible for the movement of planets around the sun.",
]

# Encode the queries and documents. Note that queries benefit from using a prompt
# Here we use the prompt called "query" stored under `model.prompts`, but you can
# also pass your own prompt via the `prompt` argument
query_embeddings = model.encode(queries, prompt_name="query")
document_embeddings = model.encode(documents)

# Compute the (cosine) similarity between the query and document embeddings
similarity = model.similarity(query_embeddings, document_embeddings)
print(similarity)
# tensor([[0.7646, 0.1414],
#         [0.1355, 0.6000]])


tensor([[0.7646, 0.1414],
        [0.1355, 0.6000]])


In [7]:
query_embeddings @ document_embeddings.T

array([[0.76455677, 0.14142507],
       [0.13549742, 0.5999553 ]], dtype=float32)

# Jina API
tasks
- retrieval.query
在查询文档检索任务中向量化查询。
- retrieval.passage
将文档向量化查询文档检索任务中。
- code.query
在代码检索任务中向量化查询。
- code.passage
在代码检索任务中向量化代码文档。
- text-matching
语义文本相似度、广义对称检索、推荐、查找相似项、去重。
- none
不使用 LoRA 适配器。返回通用向量，可用于调试或黑客攻击。

In [ ]:
import requests

url = 'https://api.jina.ai/v1/embeddings'

headers = {
    'Content-Type': 'application/json',
    'Authorization': 'Bearer xxxxxxx'  # 替换为你的API密钥
}

data = {
    "model": "jina-embeddings-v4",
    "task": "text-matching",
    "input": [
        {"text": "关闭名称为小红的灯"},
        {"text": "猫眼的红外灯可以关闭吗"},
        # {"text": "海滩上美丽的日落"},
        # {"text": "浜辺に沈む美しい夕日"}
    ]
}

response = requests.post(url, json=data, headers=headers)


In [7]:
len(response.json()['data'][0]['embedding'])

2048

In [9]:
embs1, embs2 = response.json()['data'][0]['embedding'], response.json()['data'][1]['embedding']
# 将embs1和embs2转换为numpy数组并计算向量积
import numpy as np
embs1 = np.array(embs1)
embs2 = np.array(embs2)
# 计算两个向量的点积
similarity = embs1 @ embs2.T
print(similarity)

0.8025411441940378


# Doubao seed

In [ ]:
api_key = "xxxxxx"
import os
import torch
from volcenginesdkarkruntime import Ark
from typing import Optional, List

def encode(
    client, inputs: List[str], is_query: bool = False, mrl_dim: Optional[int] = None
):
    if is_query:
        # use instruction for optimal performance, feel free to tune this instruction for different tasks
        # to reproduce MTEB results, refer to https://github.com/embeddings-benchmark/mteb/blob/main/mteb/models/seed_models.py for detailed instructions per task)
        inputs = [
            f"Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: {i}".format(
                i
            )
            for i in inputs
        ]
    resp = client.embeddings.create(
        model="doubao-embedding-large-text-250515",
        input=inputs,
        encoding_format="float",
    )
    embedding = torch.tensor([d.embedding for d in resp.data], dtype=torch.bfloat16)
    if mrl_dim is not None:
        assert mrl_dim in [2048, 1024, 512, 256]
        embedding = embedding[:, :mrl_dim]
    # normalize to compute cosine sim
    embedding = torch.nn.functional.normalize(embedding, dim=1, p=2).float().numpy()
    return embedding


# gets API Key from environment variable ARK_API_KEY
client = Ark(
    # api_key = os.getenv("ARK_API_KEY"),
    api_key = api_key,
    base_url="https://ark.cn-beijing.volces.com/api/v3",
)

print("----- embeddings -----")
inputs = ["关闭名称为小红的灯", "猫眼的红外灯可以关闭吗"]
embedding = encode(client, inputs, is_query=False, mrl_dim=1024)
print(embedding)

----- embeddings -----
[[-0.0090332   0.06201172 -0.03369141 ... -0.00634766  0.03491211
   0.01263428]
 [-0.00582886  0.02832031 -0.00506592 ... -0.00111389  0.02087402
  -0.00866699]]


In [19]:
embedding[0] @ embedding[1].T

0.71048594

In [ ]:
from volcenginesdkarkruntime import Ark

client = Ark(
    api_key = "xxxxxx",
    base_url="https://ark.cn-beijing.volces.com/api/v3",
)

print("----- multimodal embeddings request -----")
resp = client.multimodal_embeddings.create(
    model="doubao-embedding-vision-250615",
    input=[
        {
            "type":"text",
            "text":"天很蓝，海很深"
        },
        {
            "type": "image_url",
            "image_url": {
                "url": "https://ark-project.tos-cn-beijing.volces.com/images/view.jpeg"
            }
        }
    ]
)
print(resp)

In [6]:
# 获取resp的所有字段或属性
print("----- multimodal embeddings response -----")
for field in resp.__dict__:
    print(f"{field}: {getattr(resp, field)}")

----- multimodal embeddings response -----
id: 0217525628096583ea2b85053c777351b2ab3be881b0ef8209a06
created: 1752562810
data: {'embedding': [-0.036376953125, -0.037841796875, 0.01544189453125, 0.050048828125, -0.021728515625, -0.005950927734375, 0.0228271484375, -0.039306640625, -0.0230712890625, 0.0166015625, -0.01318359375, -0.0108642578125, -0.0277099609375, 0.109375, 0.0341796875, 0.022216796875, -0.0206298828125, -0.0283203125, -0.034912109375, 0.025146484375, -0.0218505859375, -0.0111083984375, 0.0004444122314453125, -0.017578125, 0.0240478515625, -0.0194091796875, 0.000614166259765625, -0.00274658203125, -0.0289306640625, 0.01092529296875, -0.038818359375, 0.00860595703125, 0.01416015625, -0.0289306640625, 0.002288818359375, 0.04345703125, 0.02197265625, 0.0478515625, -0.025634765625, 0.022705078125, -0.017822265625, 0.01422119140625, -0.0380859375, -0.01312255859375, -0.00518798828125, 0.00193023681640625, 0.024658203125, -0.0068359375, -0.00726318359375, -0.021240234375, -0.0